***Create and Inspect a Tensor***

In [ ]:
import torch

def make_tensor():
	"""Return a 2x3 float32 tensor [[1, 2, 3], [4, 5, 6]]."""
	# TODO
	return torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

t = make_tensor()
print(t)
print("----------------")
t = make_tensor()
print(tuple(t.shape))
print("----------------")
t = make_tensor()
print(float(t[1, 2].item()))

tensor([[1., 2., 3.],
        [4., 5., 6.]])
----------------
(2, 3)
----------------
6.0


***Reshape and Transpose a Tensor***

In [ ]:
import torch

def reshape_transpose(t):
	"""Reshape a 1D tensor of 6 elements to 2x3 (row-major) and return its transpose.

	Args:
		t (torch.Tensor): 1D tensor with exactly 6 elements.

	Returns:
		torch.Tensor: Transpose of the 2x3 reshape, with shape (3, 2).
	"""
	# TODO: reshape to (2, 3) then transpose to (3, 2)
	return t.reshape(2, 3).T   # or use .t() 

print(reshape_transpose(torch.tensor([1, 2, 3, 4, 5, 6])))
print("---------------------")
print(reshape_transpose(torch.tensor([1., 2., 3., 4., 5., 6.])))


tensor([[1, 4],
        [2, 5],
        [3, 6]])
---------------------
tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])


***Gradient of a Square with Autograd***

In [ ]:
import torch

def grad_of_square(x_val):
	
	# TODO: create tensor, compute y = x**2, backward, return grad
	x = torch.tensor(x_val, requires_grad=True)
	y = x ** 2
	y.backward()
	return x.grad.item()
  

print(grad_of_square(3.0))

***Gradient of a Weighted Sum of Squares***

In [ ]:
import torch

def grad_wss(w_list, x_list):
	"""Build w (requires_grad) and x from lists, compute
	loss = 0.5 * sum((w * x)**2), backward, return w.grad
	as a list of floats rounded to 4 decimals.
	"""
	# TODO
	w = torch.tensor(w_list, requires_grad=True)
	x = torch.tensor(x_list)
	L = 1/2 * ((w * x)**2).sum()
	L.backward()
	return [round(g, 4) for g in w.grad.tolist()]

print(grad_wss([1.0, 2.0], [3.0, 4.0]))

[9.0, 32.0]


***Lab : Fit Linear Regression with Autograd***

In [ ]:
import torch
import numpy as np


def fit_linear_regression(X, y, lr=0.1, steps=500):
	
	w = torch.zeros(X.shape[1], requires_grad=True)
	b = torch.tensor(0.0, requires_grad=True)

	for _ in range(steps):
		y_pred = X @ w + b 
		loss = torch.mean((y_pred - y)**2)
		loss.backward()
		# manual GD update under torch.no_grad()
		with torch.no_grad():
			w -= lr * w.grad
			b -= lr * b.grad
		# zero gradients
		w.grad.zero_()
		b.grad.zero_()

	# return detached w, b
	return w.detach(), b.detach()
	


***Single Linear Neuron Forward***

In [ ]:
import torch
import torch.nn as nn

def single_neuron_forward(x):
	# create the layer
	neuron = nn.Linear(3, 1)
	
	# fix the weights inside torch.no_grad()
	with torch.no_grad():
		neuron.weight.copy_(torch.tensor([[0.5, -0.2, 0.3]]))
		neuron.bias.copy_(torch.tensor([0.1]))
	
	# run forward and return as float
	return float(neuron(x).item())

print(single_neuron_forward(torch.tensor([[1.0, 1.0, 1.0]])))
print("----------------------------")
print(single_neuron_forward(torch.tensor([[2.0, 0.0, 0.0]])))



0.7000000476837158
----------------------------
1.100000023841858


***Two-Layer MLP Forward Pass***

In [ ]:
import torch
import torch.nn as nn


def two_layer_mlp_forward(x, w1, b1, w2, b2):

	model = nn.Sequential(
		nn.Linear(2, 2), # index 0
		nn.ReLU(),       # index 1
		nn.Linear(2, 1)  # index 2 
	)

	with torch.no_grad():
		model[0].weight.copy_(w1)  
		model[0].bias.copy_(b1)

		model[2].weight.copy_(w2)
		model[2].bias.copy_(b2)

	out = model(x)
	return out.item()

x = torch.tensor([[1.0, -1.0]])
w1 = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b1 = torch.tensor([0.0, 0.0])
w2 = torch.tensor([[1.0, 1.0]])
b2 = torch.tensor([0.0])
print(two_layer_mlp_forward(x, w1, b1, w2, b2))

1.0


***Implement ReLU and Leaky ReLU***

In [ ]:
import torch

def relu(t):
	"""Element-wise ReLU: max(0, t).

	Args:
		t (torch.Tensor): input tensor

	Returns:
		torch.Tensor: activated tensor
	"""
	# TODO: implement with pure torch ops
	return torch.clamp(t, min=0)

def leaky_relu(t, slope=0.01):
	"""Element-wise Leaky ReLU with given negative slope.

	Args:
		t (torch.Tensor): input tensor
		slope (float): slope for negative values

	Returns:
		torch.Tensor: activated tensor
	"""
	# TODO: implement with pure torch ops
	return torch.where(t > 0, t, slope * t)


***Numerically Stable Softmax***

In [ ]:
import torch

def softmax(t, dim):
	"""Numerically stable softmax along dim.

	Args:
		t (torch.Tensor): input tensor
		dim (int): dimension along which to apply softmax

	Returns:
		torch.Tensor: tensor of same shape as t; slices along dim sum to 1
	"""
	# TODO: subtract max along dim, exp, then normalize
	max_val = t.max(dim = dim, keepdim = True).values
	shifted = t - max_val
	exp = torch.exp(shifted)
	sum_exp = exp.sum(dim = dim, keepdim= True)
	return exp / sum_exp

t = torch.tensor([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]])
result = softmax(t, dim=1)
print(torch.round(result * 1e4) / 1e4)
print("-------------")
t = torch.tensor([[1000.0, 1001.0, 1002.0]])
result = softmax(t, dim=1)
print(torch.round(result * 1e4) / 1e4)

tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])
-------------
tensor([[0.0900, 0.2447, 0.6652]])


***Mean Squared Error from Scratch***

In [ ]:
import torch

def mse(pred, target):
	"""
	Compute mean squared error between pred and target.

	Args:
		pred (torch.Tensor): Predicted values.
		target (torch.Tensor): Ground-truth values (same shape as pred).

	Returns:
		float: Mean of squared differences.
	"""
	# TODO
	mse = torch.mean((pred - target)**2)
	return  mse.item()

print(mse(torch.tensor([0.0, 0.0]), torch.tensor([1.0, 0.0])))
print(mse(torch.tensor([5.0]), torch.tensor([2.0])))



0.5
9.0


***Binary Cross-Entropy from Logits***

In [ ]:
import torch

def bce_with_logits(logits, targets):
	loss = (
		torch.clamp(logits, min=0)
		- logits * targets
		+ torch.log(1 + torch.exp(-torch.abs(logits)))
	)

	loss = torch.mean(loss)
	return round(loss.item(), 4)

print(bce_with_logits(torch.tensor([2.0, -2.0]), torch.tensor([1.0, 0.0])))


0.1269


***Dropout in Train vs Eval Mode***

In [ ]:
import torch
import torch.nn as nn


def dropout_demo():
	"""Demonstrate Dropout behavior in eval vs train mode.

	Returns:
		tuple: (eval_output, train_nonzero_count)
			eval_output: result of Dropout(ones) in eval mode (identity)
			train_nonzero_count: int count of nonzero elements after Dropout in train mode
	"""
	# TODO: seed, ones(10), Dropout(0.5), eval output, train nonzero count
	torch.manual_seed(0)
	X = torch.ones(10)

	drop = nn.Dropout(p = 0.5)

	# Evaluation mode 
	drop.eval()
	eval_output = drop(X)

	# Training mode 
	drop.train()
	train_output = drop(X)

	# Count 
	count_nonzeros = torch.count_nonzero(train_output).item()

	return eval_output , count_nonzeros

print(dropout_demo()[1])
print("----------")
out, cnt = dropout_demo()
print(tuple(out.shape))
print(out.dtype)
print(type(cnt).__name__)

4
----------
(10,)
torch.float32
int


***BatchNorm1d Forward in Eval Mode***

In [ ]:
import torch

def bn_eval(x, mean, var, gamma, beta, eps=1e-5):
	"""Apply batch-norm inference normalization.

	Args:
		x (Tensor): input tensor
		mean (Tensor): running mean
		var (Tensor): running variance
		gamma (Tensor): scale parameter
		beta (Tensor): shift parameter
		eps (float): numerical stability constant

	Returns:
		Tensor: normalized and affine-transformed tensor
	"""
	# TODO
	normalized =( x - mean ) /  (torch.sqrt(var + eps))
	out = gamma * normalized + beta
	return out

print(torch.round(bn_eval(torch.tensor([3.0, 7.0]), torch.tensor(5.0), torch.tensor(4.0), torch.tensor(3.0), torch.tensor(-1.0))*1e4)/1e4)

tensor([-4.,  2.])


***Count Parameters of a Sequential Model***

In [ ]:
import torch
import torch.nn as nn

def count_params():
	"""Build Sequential(Linear(4,8), ReLU, Linear(8,2)) and return trainable param count.

	Returns:
		int: total number of trainable parameters
	"""
	# TODO: build the model and sum p.numel() for trainable params
	model = nn.Sequential(
		nn.Linear(4, 8),  # weight.shape = (4, 8) and bias.shape = (8, ) || total shape = 4*8 + 8 = 32 + 8 = 40
		nn.ReLU(),
		nn.Linear(8, 2)   # weight.shape = (8, 2) and bias.shape = (2, ) || total shape = 8*2 + 2 = 16 + 2 = 18 
	)

	total = 0 

	for p in model.parameters():
		if p.requires_grad :
			total += p.numel()
	return total

print(count_params())

58


***Lab : MLP with Dropout and BatchNorm***

***Conv2d Output Shape***

In [1]:
def conv_out_shape(h, w, kernel, stride, padding):
    # // Floor Division 
    h_out = (h + 2 * padding - kernel) // stride + 1
    w_out = (w + 2 * padding - kernel) // stride + 1

    return (h_out, w_out)

print(conv_out_shape(28, 28, 5, 2, 0))
print('---------------')
print(conv_out_shape(16, 32, 3, 2, 1))

(12, 12)
---------------
(8, 16)
